### Negation adaptor
Question: The paper tests mainly on verbal negation ("is not," "did not") rather than negation through prefixes like "without," "-less," "non-" etc. How to overcome limitation?

**Strategy 1: Direct Pattern Extension (Simplest)**  
Hypothesis: If verbal negation weights work, attribute negation might already activate similar dimensions.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
import random

In [2]:

model = SentenceTransformer('intfloat/e5-small-v2')

# Test if different negation types activate similar embedding dimensions
negation_patterns = {
    'verbal': [
        ('passage: The jacket is warm', 'passage: The jacket is not warm'),
        ('passage: The coat has insulation', 'passage: The coat does not have insulation'),
    ],
    'prefix': [
        ('passage: comfortable fabric', 'passage: uncomfortable fabric'),
        ('passage: waterproof jacket', 'passage: non-waterproof jacket'),
    ],
    'suffix': [
        ('passage: jacket with sleeves', 'passage: sleeveless jacket'),
        ('passage: coat with feathers', 'passage: featherless coat'),
    ],
    'prepositional': [
        ('passage: jacket with hood', 'passage: jacket without hood'),
        ('passage: shirt with pockets', 'passage: shirt without pockets'),
    ]
}



In [3]:

# Analyze dimension activations
def analyze_negation_dimensions(model, pattern_dict):
    results = {}
    
    for pattern_type, pairs in pattern_dict.items():
        dimension_diffs = []
        
        for positive, negative in pairs:
            emb_pos = model.encode(positive)
            emb_neg = model.encode(negative)
            
            # Dimension-wise difference
            diff = np.abs(emb_pos - emb_neg)
            dimension_diffs.append(diff)
        
        # Average across pairs
        avg_diff = np.mean(dimension_diffs, axis=0)
        results[pattern_type] = avg_diff
    
    return results

dimensions = analyze_negation_dimensions(model, negation_patterns)

# Check correlation between verbal and attribute negation dimensions
from scipy.stats import pearsonr

corr_prefix = pearsonr(dimensions['verbal'], dimensions['prefix'])[0]
corr_suffix = pearsonr(dimensions['verbal'], dimensions['suffix'])[0]
corr_prep = pearsonr(dimensions['verbal'], dimensions['prepositional'])[0]

print(f"Verbal vs Prefix correlation: {corr_prefix:.3f}")
print(f"Verbal vs Suffix correlation: {corr_suffix:.3f}")
print(f"Verbal vs Prepositional correlation: {corr_prep:.3f}")

Verbal vs Prefix correlation: 0.306
Verbal vs Suffix correlation: 0.246
Verbal vs Prepositional correlation: 0.298


### Strategy 2 (Mixed-Pattern Training)
Since correlations are low, you must train with attribute negation data.

**Step 1: Generate Mixed-Pattern Training Data**

In [ ]:
def generate_attribute_negation_triplets(product_attributes, target_count=250):
    """
    Generate triplets focusing on attribute negation patterns
    """
    triplets = []
    
    for category, data in product_attributes.items():
        items = data['items']
        attributes = data['attributes']
        
        for item in items:
            for attr in attributes:
                # Template 1: Prepositional negation (40% - most common in products)
                triplets.append((
                    f"passage: {item} with {attr}",
                    f"passage: {attr} {item}",
                    f"passage: {item} without {attr}"
                ))
                
                # Template 2: Suffix negation (35% - common for adjectives)
                if attr in ['hood', 'sleeves', 'pockets', 'collar', 'buttons', 'feathers', 'fur']:
                    triplets.append((
                        f"passage: {item} with {attr}",
                        f"passage: {attr}ed {item}",  # hooded, sleeved, etc.
                        f"passage: {attr}less {item}"
                    ))
                
                # Template 3: Prefix negation (15% - for properties)
                if attr in ['waterproof', 'insulated', 'reversible', 'breathable', 'stretch']:
                    triplets.append((
                        f"passage: {attr} {item}",
                        f"passage: {item} that is {attr}",
                        f"passage: non-{attr} {item}"
                    ))
                
                # Template 4: Alternative suffix (-free)
                if attr in ['leather', 'cotton', 'wool', 'fur', 'feathers', 'down']:
                    triplets.append((
                        f"passage: {item} with {attr}",
                        f"passage: {attr} {item}",
                        f"passage: {attr}-free {item}"
                    ))
    
    # Shuffle and sample
    random.shuffle(triplets)
    return triplets[:target_count]

In [ ]:

# Your domain-specific attributes
product_attributes = {
    'clothing': {
        'items': ['jacket', 'coat', 'sweatshirt', 'hoodie', 'shirt', 'pants', 'dress', 'sweater'],
        'attributes': ['hood', 'pockets', 'sleeves', 'collar', 'buttons', 'zipper', 'lining', 'belt']
    },
    'materials': {
        'items': ['jacket', 'coat', 'bag', 'shoes', 'belt'],
        'attributes': ['leather', 'cotton', 'wool', 'silk', 'synthetic', 'fur', 'feathers', 'down']
    },
    'features': {
        'items': ['jacket', 'coat', 'pants', 'bag'],
        'attributes': ['waterproof', 'insulated', 'reversible', 'breathable', 'stretch', 'padded']
    }
}

# Generate training data
training_triplets = generate_attribute_negation_triplets(product_attributes, target_count=250)
print(f"Generated {len(training_triplets)} training triplets")

# Display sample
print("\nSample triplets:")
for i, (ref, para, neg) in enumerate(training_triplets[:5]):
    print(f"\n{i+1}. Reference: {ref}")
    print(f"   Paraphrase: {para}")
    print(f"   Negation: {neg}")